In [25]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import os
import base64
from datetime import datetime

# --- FILE SETUP ---
FILE = "bill_history.csv"
if os.path.exists(FILE):
    history_df = pd.read_csv(FILE)
else:
    history_df = pd.DataFrame(columns=["Date", "Bill", "Total", "Result"])

# --- CSS FIXES & UI STYLING ---
style = HTML("""
    <style>
        .widget-label { min-width: 160px !important; }
        .summary-box {
            background-color: #f4f7f6;
            border-left: 5px solid #2e7d32;
            padding: 12px;
            margin-bottom: 15px;
            border-radius: 4px;
            font-family: sans-serif;
        }
        .download-btn {
            cursor:pointer; background-color:#f0f0f0; border:1px solid #dcdcdc; 
            padding:5px 12px; border-radius:4px; font-size: 13px; color: #333;
        }
    </style>
""")

In [26]:
# Create Input Widgets
title_input = widgets.Text(description="Bill Name:", placeholder="e.g. Electric Bill")
andrew_input = widgets.FloatText(description="Andrew Paid ($):", value=0.0)
moi_input = widgets.FloatText(description="Moi Paid ($):", value=0.0)
calc_button = widgets.Button(description="Calculate & Save", button_style='success')

# Placeholders for dynamic content
summary_output = widgets.Output()
download_output = widgets.Output()
output_area = widgets.Output()

# Arrange everything in a vertical layout
layout = widgets.VBox([
    widgets.HTML("<h2>⚖️ Andrew & Moi: Bill Balancer</h2>"),
    summary_output, 
    title_input, 
    andrew_input, 
    moi_input, 
    widgets.HBox([calc_button, download_output]), 
    output_area
])

In [27]:
def display_summary():
    """Calculates and displays the total spent for the current month."""
    with summary_output:
        clear_output()
        if not history_df.empty:
            current_month = datetime.now().strftime("%Y-%m")
            
            temp_df = history_df.copy()
            # The 'r' before the string fix:
            temp_df['Total_Num'] = temp_df['Total'].replace(r'[\$,]', '', regex=True).astype(float)
            
            monthly_data = temp_df[temp_df['Date'].str.startswith(current_month)]
            monthly_total = monthly_data['Total_Num'].sum()
            count = len(monthly_data)
            
            display(HTML(f'''
                <div class="summary-box">
                    <strong>📊 {datetime.now().strftime("%B %Y")} Summary</strong><br>
                    Total Spent: ${monthly_total:.2f} across {count} entries.
                </div>
            '''))

def create_download_link():
    if os.path.exists(FILE):
        df = pd.read_csv(FILE)
        formatted_df = df.rename(columns={"Date": "Date Settled", "Bill": "Description", "Total": "Total Amount", "Result": "Settlement Status"})
        csv_data = formatted_df.to_csv(index=False)
        b64 = base64.b64encode(csv_data.encode()).decode()
        filename = f"Bill_Balancer_{datetime.now().strftime('%Y_%m_%d')}.csv"
        return HTML(f'<a href="data:file/csv;base64,{b64}" download="{filename}" style="text-decoration:none;"><button class="download-btn">📥 Download for Excel</button></a>')
    return HTML("")

def on_click(b):
    global history_df
    with output_area:
        clear_output()
        total = andrew_input.value + moi_input.value
        share = total / 2
        
        if andrew_input.value > share:
            res = f"Moi owes Andrew ${andrew_input.value-share:.2f}"
        elif moi_input.value > share:
            res = f"Andrew owes Moi ${moi_input.value-share:.2f}"
        else:
            res = "Even split!"
        
        new_row = {"Date": datetime.now().strftime("%Y-%m-%d"), "Bill": title_input.value, "Total": f"${total:.2f}", "Result": res}
        history_df = pd.concat([history_df, pd.DataFrame([new_row])], ignore_index=True)
        history_df.to_csv(FILE, index=False)
        
        print(f"✅ Saved! {res}")
        display(history_df.tail(3))
        display_summary()
        with download_output:
            clear_output(); display(create_download_link())

calc_button.on_click(on_click)
display(style)
display(layout)
display_summary()